# M04C: Advanced Few-Shot Techniques

Go beyond simple classification. Use few-shot for complex data extraction, format conversion, and style transfer.

**Topics:**
- Few-shot for data extraction & format conversion
- Combining few-shot with instructions (The Power Combo)
- Dynamic example selection

---

## 🔧 Step 1: Setup

In [ ]:
import os
from pathlib import Path
from dotenv import load_dotenv
import openai

load_dotenv(dotenv_path=Path("..") / ".env")

client = openai.OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
MODEL = "gpt-5-mini"


def truncate_response(text, max_length=1200):
    """Truncate text for readability."""
    if len(text) <= max_length:
        return text
    return text[:max_length] + f"...\n\n💡 (Truncated from {len(text)} chars)"


print(f"✅ Setup complete: Using {MODEL}!")

---

## 📖 Quick Review from Module 2

M02B covered: 3-5 examples as the sweet spot, diminishing returns after 5-7 examples, and sentiment classification.

Now we go beyond classification into complex production tasks.

---

## 📊 Complex Task 1: Data Extraction

Extract multiple structured fields from unstructured text.

In [ ]:
print("📊 COMPLEX DATA EXTRACTION")
print("="*60)

extraction_prompt = """Extract action items from meeting notes:

Notes: "John agreed to send the report by Friday. 
Sarah will schedule the follow-up for next Tuesday."
Action Items:
1. Person: John | Task: Send report | Deadline: Friday
2. Person: Sarah | Task: Schedule follow-up | Deadline: next Tuesday

Notes: "The team decided to push the launch date. 
Mike needs to update the roadmap ASAP."
Action Items:
1. Person: Mike | Task: Update roadmap | Deadline: ASAP

Notes: "Lisa and Tom will review the designs tomorrow. 
Alex will email the client tonight."
Action Items:"""

response = client.responses.create(
    model=MODEL,
    input=extraction_prompt,
    instructions="Extract action items from examples. Be concise."
)
print(truncate_response(response.output_text))
print("="*60)

### 🔑 Why This Works

Without examples, the model invents its own format. With examples, it copies yours exactly.

---

## 🔄 Complex Task 2: Format Conversion

Transform data between formats with precise schemas.

In [ ]:
print("🔄 FORMAT CONVERSION")
print("="*60)

format_prompt = """Convert to JSON:

Product: iPhone 15 Pro (256GB) - In stock - $1,099
JSON:
{"name": "iPhone 15 Pro", "storage": "256GB", "available": true, "price": 1099}

Product: Samsung S24 Ultra, 512GB, Out of stock, Price: $1299
JSON:
{"name": "Samsung S24 Ultra", "storage": "512GB", "available": false, "price": 1299}

Product: Google Pixel 8 - 128GB - Available Now - $699
JSON:"""

response = client.responses.create(
    model=MODEL,
    input=format_prompt,
    instructions="Convert to JSON format shown in examples. No code fences. Be concise."
)
print(truncate_response(response.output_text))
print("="*60)

### 🔑 Why This Works

Examples demonstrated exact JSON schema, boolean conversion ("In stock" → true), and field mapping.

**Note:** Few-shot JSON works for demos, but the model can still add extra fields or break the schema. For strict compliance, see structured outputs (M03A).

---

## 🎨 Complex Task 3: Style Transfer

Change the tone or complexity of text while preserving the exact meaning.

In [ ]:
print("🎨 STYLE TRANSFER")
print("="*60)

style_prompt = """Simplify technical explanations:

Technical: "The API utilizes a rate-limiting algorithm to prevent DDoS attacks."
Simple: "The system slows down traffic to stop attackers from crashing it."

Technical: "Data persistence is achieved via synchronous replication to off-site nodes."
Simple: "We save your data in two places at once so you never lose it."

Technical: "The latent space representation allows for semantic interpolation between concepts."
Simple:"""

response = client.responses.create(
    model=MODEL,
    input=style_prompt,
    instructions="Simplify the technical text using the style from examples. Be concise."
)
print(truncate_response(response.output_text))
print("="*60)

### 🔑 Why This Works

The examples define what "simplify" means — the model copies the pattern, not just the instruction.

---

## 🤝 Combining Few-Shot + Instructions

Examples show WHAT (concrete patterns), instructions handle WHY and edge cases.

In [ ]:
print("🤝 COMBINING FEW-SHOT + INSTRUCTIONS")
print("="*60)

few_shot_prompt = """Classify customer sentiment:

"I love this product!" → positive
"The screen is broken." → negative
"It arrived today." → neutral

"I don't hate it, but I don't love it either."""

edge_case_instructions = """Classify sentiment using examples.
Edge case handling:
- If mixed sentiment, lean toward the dominant tone
- If unclear, classify as 'neutral'
- Be concise.
"""

response = client.responses.create(
    model=MODEL,
    input=few_shot_prompt,
    instructions=edge_case_instructions
)
print(truncate_response(response.output_text))

### 🔑 Why This Works

**Problem:** Examples can't cover every edge case.  
**Solution:** Examples handle standard cases, instructions handle edge cases.

---

## 🎲 Dynamic Example Selection

Sending all examples wastes tokens. Solution: select the most relevant examples based on the input.

In [ ]:
SENTIMENT_EXAMPLE_POOL = {
    'product': [
        ('This phone is incredible!', 'positive'),
        ('The quality is terrible.', 'negative'),
        ('The phone has a 6.1 inch screen.', 'neutral')
    ],
    'service': [
        ('Customer support was amazing!', 'positive'),
        ('Waited 2 hours for help.', 'negative'),
        ('I spoke with support on Monday.', 'neutral')
    ],
    'shipping': [
        ('Arrived super fast!', 'positive'),
        ('Package was damaged.', 'negative'),
        ('The package shipped from Texas.', 'neutral')
    ]
}


def select_relevant_examples(text, example_pool):
    if any(word in text.lower() for word in ['product', 'phone', 'quality', 'device']):
        return example_pool['product']
    elif any(word in text.lower() for word in ['support', 'service', 'help', 'staff']):
        return example_pool['service']
    elif any(word in text.lower() for word in ['shipping', 'delivery', 'package', 'arrived']):
        return example_pool['shipping']
    else:
        return example_pool['product']


def build_dynamic_prompt(text, examples):
    prompt = 'Classify sentiment:\n\n'
    for input_text, label in examples:
        prompt += f'Input: {input_text}\nSentiment: {label}\n---\n'
    prompt += f'Input: {text}\nSentiment:'
    return prompt


# --------------------------------------------------------------
print('✅ Dynamic example selection ready!')

### 🧪 Testing Dynamic Selection

In [ ]:
print("🔬 DYNAMIC EXAMPLE SELECTION DEMO")
print("="*60)

feedback_cases = [
    "The phone screen is beautiful!",
    "Support team was really helpful",
    "Package arrived damaged yesterday"
]

for feedback in feedback_cases:
    examples = select_relevant_examples(feedback, SENTIMENT_EXAMPLE_POOL)
    prompt = build_dynamic_prompt(feedback, examples)
    
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        instructions="Use examples to classify. Be concise."
    )
    
    print(f"\n📝 Input: {feedback}")
    print(f"📊 Result: {response.output_text.strip()}")

print("="*60)

### 🔑 Why This Works

Relevant examples → better results, fewer tokens, scales easily.

---

### 💪 Your Turn: Build Production Email Classifier

Combine all techniques: example pool, dynamic selection, complex extraction, and instructions for edge cases.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise: Build Production Email Classifier
# --------------------------------------------------------------
# Objective: Combine dynamic selection, complex extraction,
# and instructions.

EMAIL_EXAMPLES = {
    'urgent': [
        ('I cannot log in and have a demo in 5 minutes!',
         'Priority: High | Topic: Login Issue | Response: Immediate'),
        ('System is down for all users.',
         'Priority: High | Topic: Outage | Response: Immediate')
    ],
    'feedback': [
        ('Great job on the new update.',
         'Priority: Low | Topic: Feedback | Response: Optional'),
        ('I found a small typo on the dashboard.',
         'Priority: Low | Topic: Bug Report | Response: Standard')
    ]
}

print("📧 PRODUCTION CLASSIFIER EXERCISE")
print("=" * 60)

# TODO 1: Define a selector function
# Copy your select_dynamic_examples() function
# from the previous exercise, then adapt it.
# Hints:
# - "urgent", "down", "login" -> urgent
# - "typo", "great", "update" -> feedback
def select_email_examples(text):
    pass

# TODO 2: Define the classifier function
# Copy your build_dynamic_prompt() function
# from the previous exercise, then adapt it.
# Hint: Use select_email_examples() to pick
# examples, then include them in the prompt.
def classify_email(text):
    pass

# TODO 3: Test with different emails


---

## 🎯 Key Takeaways

**🧩 Beyond Classification (positive, negative, neutral):**
- Few-shot works for extraction, conversion, and style transfer
- Examples enforce schemas better than instructions alone
- Examples demonstrate tone better than describing it

**🤝 Few-Shot + Instructions:**
- Examples show concrete patterns, instructions handle edge cases
- Use when examples alone can't cover all scenarios
- **The Flow:** Example pool → Select relevant → Add instructions → Test

**🎲 Dynamic Example Selection:**
- Send relevant examples, not all examples
- Keyword matching is simple but effective for routing
- Scales to large example pools while keeping prompts cheap

---

### 📍 Next Step

**M04D: Self-Consistency & Output Quality** — Generate multiple responses, pick the best through voting or quality scoring.

---

## 🔧 Troubleshooting

**Model ignoring examples?**
- Put examples above the new input in the prompt
- Ensure examples exactly match desired output format
- Reduce number of examples if model gets confused

**Extraction missing fields?**
- Add examples showing how to handle missing data (e.g., "N/A")
- Explicitly list required fields in instructions

**Not sure if you need few-shot?**
- ✅ Complex logic, strict formatting, nuanced style
- ❌ Simple tasks (waste of tokens)

**Dynamic selection not picking right examples?**
- Keyword matching can be brittle — test with varied inputs
- Ensure example categories are distinct

**Still having issues?**
- Copy any error message and paste it into ChatGPT, Claude, Gemini, or Grok — they're great at debugging
- Re-watch the lecture for this module
- Post to the Q&A with your error message and output

---